# HiP-VesselAdapt on Colab: `colab_tpu`

Runtime → Change runtime type → **TPU**. Then run the cells in order. Cells 1 and 2 need you (Google Drive access; your `kaggle.json` from kaggle.com → Settings → API). Training state is saved on Drive after every epoch, so running the notebook again later resumes the run.

Job list: `kaggle_kernels/jobs/colab_tpu.json`; generated by `kaggle_kernels/colab/make_colab_notebook.py`.

In [ ]:
# 1. Google Drive (outputs and resume state)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/hip-vesseladapt/colab_tpu'

In [ ]:
# 2. Kaggle API key: choose your kaggle.json
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
# 3. Inputs (code, SAM-Med3D weights, ROIs, training kidneys)
import os, subprocess, glob
os.environ["KAGGLE_CONFIG_DIR"] = "/root/.kaggle"
def sh(c):
    print("$", c, flush=True)
    subprocess.run(c, shell=True, check=True)
sh("pip install -q kaggle")
I = "/kaggle/input"
sh(f"mkdir -p {I}/datasets/rabeya100x/hip-vesseladapt-code {I}/datasets/rabeya100x/sam-med3d-weights {I}/rois {I}/blood-vessel-segmentation")
sh(f"kaggle datasets download rabeya100x/hip-vesseladapt-code -p {I}/datasets/rabeya100x/hip-vesseladapt-code --unzip -q")
sh(f"kaggle datasets download rabeya100x/sam-med3d-weights -p {I}/datasets/rabeya100x/sam-med3d-weights --unzip -q")
sh(f"kaggle kernels output rabeya100x/hip-vesseladapt-rois -p {I}/rois -q")
# only the training kidneys (images of kidney_3_dense live in kidney_3_sparse)
if not os.path.isdir(f"{I}/blood-vessel-segmentation/train/kidney_1_dense"):
    sh(f"kaggle competitions download -c blood-vessel-segmentation -p /content -q")
    sh(f"cd {I}/blood-vessel-segmentation && unzip -q -o /content/blood-vessel-segmentation.zip "
       "'train/kidney_1_dense/*' 'train/kidney_2/*' 'train/kidney_3_sparse/*' 'train/kidney_3_dense/*' && rm /content/blood-vessel-segmentation.zip")
sh("df -h /content | tail -1; free -g | head -2; nproc")

In [ ]:
# 4. PyTorch/XLA (TPU runtime only)
# PyTorch/XLA for the TPU runtime (same versions as the Kaggle TPU image)
import subprocess
subprocess.run("pip install -q torch==2.8.0 'torch_xla[tpu]==2.8.0'", shell=True, check=True)
subprocess.run("python -c \"import torch_xla, torch; print(torch.__version__, torch_xla.device())\"", shell=True, check=True)

In [ ]:
%%writefile /content/kernel.py
# 5. Job runner (kaggle_kernels/kaggle_jobs_kernel.py with the job list filled in)
"""
Kaggle GPU kernel that runs a list of jobs on both GPUs of a "T4 x2" session,
one queue per GPU, so a session hour gives two GPU hours of work.

A job is a list of shell commands run in the project folder with
CUDA_VISIBLE_DEVICES set to its GPU. Typical jobs:
  - train one configuration with train_lora_v2.py, then write its benchmark
    predictions (benchmark/infer.py) for the validation and test ROIs;
  - write benchmark predictions for an existing checkpoint (first-round
    models, SAM-Med3D without fine-tuning).
Scoring (benchmark/score.py) is CPU-only and is done afterwards from the saved
predictions, so it does not use GPU time.

Inputs attached to the kernel (found by marker file, not by path):
  - code dataset (marker: train_lora_v2.py)
  - competition data blood-vessel-segmentation (marker: train/kidney_1_dense)
  - output of kernel hip-vesseladapt-100ep-gpu-train (marker: sam_med3d.pth);
    it has the SAM-Med3D checkpoint and the first-round models
  - optional: ROI dataset (marker: manifest.json next to kidney_6/)
Everything is written to /kaggle/working/out.
"""

import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile

JOBS_GPU0 = []  # filled in by the launcher (kaggle_kernels/launch_jobs.py)
JOBS_GPU1 = []
KEEP_STATE = True  # launcher --keep-state: keep state.pth so a later session can resume the training
JOBS_TPU = [
 [
  {
   "name": "train_s_aug_long_s2",
   "cmds": [
    "python train_lora_v2.py --device xla --model spacing --spacing-encoding log --fov-scale 0.25 2.0 --patch 128 --fg-prob 0.5 --refine --epochs 150 --iters 250 --val-every 5 --workers 6 --raw-tiff-root $COMP_TRAIN --val-roi-dir $VAL_ROI_DIR --time-limit-min 600 --seed 2 --out $OUT/checkpoints/s_aug_long_s2"
   ]
  }
 ]
]  # TPU sessions: one queue per TPU chip (up to 8 on a v5e-8), each job pinned to its chip
ON_TPU = os.environ.get("PJRT_DEVICE") == "TPU" or os.environ.get("ISTPUVM") == "1"

SAM_MED3D_COMMIT = "bd7d1620df0f85fade74aa9e44262de953b559f1"
CLDICE_COMMIT = "47d31a6cc4a8101b1ffe8052994821961e57af9f"
WORK_DIR = "/tmp/hip-vesseladapt"  # not under /kaggle/working, so the code is not saved with the outputs
OUT = "/kaggle/working/out"


def find_by_marker(root, is_match, max_depth=9):
    stack = [(root, 0)]
    while stack:
        path, depth = stack.pop()
        if not os.path.isdir(path):
            continue
        if is_match(path):
            return path
        if depth < max_depth:
            try:
                stack += [(os.path.join(path, n), depth + 1) for n in sorted(os.listdir(path))]
            except PermissionError:
                pass
    return None


t0 = time.time()
# only the code dataset: outputs of earlier kernels attached as inputs can hold older copies of the code
# (mounted under /kaggle/input/datasets/... on older images, /kaggle/input/<slug> on newer ones)
CODE_DIR = None
for _root in ("/kaggle/input/datasets", "/kaggle/input/hip-vesseladapt-code"):
    CODE_DIR = CODE_DIR or find_by_marker(_root, lambda p: os.path.exists(os.path.join(p, "train_lora_v2.py")))
COMP_DIR = find_by_marker("/kaggle/input", lambda p: os.path.isdir(os.path.join(p, "train", "kidney_1_dense")))
SAM_DIR = find_by_marker("/kaggle/input", lambda p: os.path.exists(os.path.join(p, "sam_med3d.pth")))
FIRST_ROUND = find_by_marker("/kaggle/input", lambda p: os.path.isdir(os.path.join(p, "checkpoints_plain_lora")))
ROI_DIR = find_by_marker("/kaggle/input", lambda p: os.path.exists(os.path.join(p, "manifest.json"))
                         and os.path.isdir(os.path.join(p, "kidney_6")))
VAL_ROI_DIR = find_by_marker("/kaggle/input", lambda p: os.path.exists(os.path.join(p, "manifest.json"))
                             and os.path.isdir(os.path.join(p, "kidney_1_dense")))
for name, d in (("code", CODE_DIR), ("competition", COMP_DIR), ("SAM-Med3D ckpt", SAM_DIR),
                ("first-round output", FIRST_ROUND), ("test ROIs", ROI_DIR), ("val ROIs", VAL_ROI_DIR)):
    print(f"[*] {name}: {d}", flush=True)
# the competition data is only needed by jobs that train or build validation ROIs
if CODE_DIR is None:
    raise RuntimeError(f"code dataset not attached; /kaggle/input has {sorted(os.listdir('/kaggle/input'))}")

os.makedirs(WORK_DIR, exist_ok=True)
shutil.copytree(CODE_DIR, WORK_DIR, dirs_exist_ok=True)
for z in ("models.zip", "utils.zip", "losses.zip", "data.zip", "benchmark.zip"):
    zp = os.path.join(WORK_DIR, z)
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(WORK_DIR)
        os.remove(zp)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "tifffile", "imagecodecs"], check=True)
for url, dst, commit in (("https://github.com/openmedlab/SAM-Med3D.git", "external/SAM-Med3D", SAM_MED3D_COMMIT),
                         ("https://github.com/jocpae/clDice.git", "external/clDice", CLDICE_COMMIT)):
    if not os.path.isdir(dst):
        subprocess.run(["git", "clone", "-q", url, dst], check=True)
    subprocess.run(["git", "-C", dst, "checkout", "-q", commit], check=True)
from utils.patch_external_repos import patch_sam_med3d
patch_sam_med3d()
os.makedirs("external/SAM-Med3D/ckpt", exist_ok=True)
ckpt = "external/SAM-Med3D/ckpt/sam_med3d.pth"
if not os.path.exists(ckpt):
    if SAM_DIR:
        os.symlink(os.path.join(SAM_DIR, "sam_med3d.pth"), ckpt)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        subprocess.run([sys.executable, "-m", "gdown", "https://drive.google.com/uc?id=1PFeUjlFMAppllS9x1kAWyCYUJM9re2Ub",
                        "-O", ckpt], check=True)

import torch
print(f"[*] torch {torch.__version__}, {torch.cuda.device_count()} GPU(s): "
      f"{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}", flush=True)
if torch.cuda.is_available():
    subprocess.run(["nvidia-smi"])

if not VAL_ROI_DIR and COMP_DIR:
    # validation ROIs from the held-out last 20 % of kidney_1_dense (seed 0, same as the ROI kernel)
    subprocess.run([sys.executable, "-m", "benchmark.rois", "--validation", "--raw-tiff-root",
                    os.path.join(COMP_DIR, "train")], check=True)
    VAL_ROI_DIR = os.path.join(WORK_DIR, "data/benchmark/val_rois")

ABLATION = find_by_marker("/kaggle/input", lambda p: os.path.isdir(os.path.join(p, "checkpoints_spacing_lora_no_cldice")))
# second-round checkpoints from earlier kernels: link every <run>/model_best.pth
# into one folder so jobs can refer to $TRAINED/checkpoints/<run>/model_best.pth
TRAINED = "/tmp/trained"
os.makedirs(os.path.join(TRAINED, "checkpoints"), exist_ok=True)
for root, dirs, files in os.walk("/kaggle/input"):
    if "model_best.pth" in files and os.path.basename(os.path.dirname(root)) == "checkpoints":
        dst = os.path.join(TRAINED, "checkpoints", os.path.basename(root))
        if not os.path.exists(dst):
            os.symlink(root, dst)
            print(f"[*] trained model: {os.path.basename(root)} ({root})", flush=True)

env_vars = {
    "ABLATION": ABLATION or "",
    "TRAINED": TRAINED,
    "COMP_TRAIN": os.path.join(COMP_DIR, "train") if COMP_DIR else "",
    "ROI_DIR": ROI_DIR or "",
    "VAL_ROI_DIR": VAL_ROI_DIR or "",
    "FIRST_ROUND": FIRST_ROUND or "",
    "OUT": OUT,
}
os.makedirs(OUT, exist_ok=True)
print(f"[*] setup took {time.time() - t0:.0f}s", flush=True)


def run_queue(gpu, jobs):
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_ALLOC_CONF="expandable_segments:True", **env_vars)
    if ON_TPU:
        # one process per chip: each sees a single chip and has its own runtime port
        env.update(TPU_VISIBLE_CHIPS=str(gpu), TPU_CHIPS_PER_PROCESS_BOUNDS="1,1,1", TPU_PROCESS_BOUNDS="1,1,1",
                   TPU_PROCESS_PORT=str(8476 + gpu), TPU_PROCESS_ADDRESSES=f"localhost:{8476 + gpu}",
                   CLOUD_TPU_TASK_ID="0", PJRT_DEVICE="TPU")
    else:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    for job in jobs:
        name = job["name"]
        log_path = os.path.join(OUT, f"{name}.log")
        t = time.time()
        with open(log_path, "w") as log:
            for cmd in job["cmds"]:
                log.write(f"$ {cmd}\n")
                log.flush()
                r = subprocess.run(cmd, shell=True, cwd=WORK_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
                if r.returncode != 0:
                    log.write(f"[!] exit code {r.returncode}\n")
                    print(f"[!] GPU{gpu} job {name}: command failed ({r.returncode}): {cmd}", flush=True)
                    break
        print(f"[✓] GPU{gpu} job {name} finished in {(time.time() - t) / 60:.1f} min", flush=True)
        with open(log_path) as f:
            tail = f.read()[-3000:]
        print(f"----- tail of {name}.log -----\n{tail}\n-----", flush=True)


n_gpu = torch.cuda.device_count()
queues = [JOBS_GPU0, JOBS_GPU1]
if ON_TPU:
    queues = JOBS_TPU or [JOBS_GPU0 + JOBS_GPU1]
elif n_gpu < 2:
    queues = [JOBS_GPU0 + JOBS_GPU1]
threads = [threading.Thread(target=run_queue, args=(g, q)) for g, q in enumerate(queues) if q]
for th in threads:
    th.start()
for th in threads:
    th.join()
# checkpoint folders can be large; keep the model weights and logs, drop the resume state
for root, _, files in ([] if KEEP_STATE else os.walk(os.path.join(OUT, "checkpoints"))):
    for f in files:
        if f in ("state.pth",):
            os.remove(os.path.join(root, f))
print(f"[✓] ALL DONE in {(time.time() - t0) / 60:.1f} min", flush=True)


In [ ]:
# 6. Run (log of each job: DRIVE_OUT/out/<job>.log)
import os
os.makedirs(DRIVE_OUT, exist_ok=True)
if not os.path.islink("/kaggle/working"):
    os.makedirs("/kaggle", exist_ok=True)
    os.symlink(DRIVE_OUT, "/kaggle/working")
os.environ["PJRT_DEVICE"] = "TPU"   # remove this line on a GPU runtime
!python /content/kernel.py 2>&1 | tee -a /kaggle/working/colab_session.log

In [ ]:
# progress
!tail -n 5 /kaggle/working/out/*.log